In [35]:
import pandas as pd
import numpy as np

In [36]:
review_path = r"C:\Users\rnu43\MyFiles\Documents\Python\PROJECTS\FRD_System\All_Beauty.jsonl\All_Beauty.jsonl"
meta_path = r"C:\Users\rnu43\MyFiles\Documents\Python\PROJECTS\FRD_System\meta_All_Beauty.jsonl\meta_All_Beauty.jsonl"

# 1. Load reviews (behavioral and text branch)
reviews_df = pd.read_json(review_path, lines=True, nrows=5000)

# 2 & 3. Stream metadata in chunks, match parent_asin to reviews_df, deduplicate, and merge
target_asins = set(reviews_df['parent_asin'].dropna().unique())
matched_meta = []

for chunk in pd.read_json(meta_path, lines=True, chunksize=10000):
  sub = chunk[chunk['parent_asin'].isin(target_asins)][
      ['parent_asin', 'price', 'average_rating', 'rating_number', 'store']
  ]
  if not sub.empty:
    matched_meta.append(sub)

meta_matched_df = (
    pd.concat(matched_meta, ignore_index=True)
    if matched_meta
    else pd.DataFrame(
        columns=['parent_asin', 'price', 'average_rating', 'rating_number', 'store']
    )
).drop_duplicates(subset=['parent_asin'])

merged_df = pd.merge(
    reviews_df, meta_matched_df, on='parent_asin', how='left'
)

print('--- Multi-Branch Dataset Ready ---')
print(f'Merged shape: {merged_df.shape}')
print(
    'Non-null metadata counts:\n',
    merged_df[['price', 'average_rating', 'rating_number', 'store']].notna().sum(),
)

--- Multi-Branch Dataset Ready ---
Merged shape: (5000, 14)
Non-null metadata counts:
 price             1159
average_rating    5000
rating_number     5000
store             4714
dtype: int64


In [37]:
print(len(merged_df))
print(len(reviews_df))
print(merged_df.shape)
print(merged_df.info())

5000
5000
(5000, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   rating             5000 non-null   int64         
 1   title              5000 non-null   object        
 2   text               5000 non-null   object        
 3   images             5000 non-null   object        
 4   asin               5000 non-null   object        
 5   parent_asin        5000 non-null   object        
 6   user_id            5000 non-null   object        
 7   timestamp          5000 non-null   datetime64[ns]
 8   helpful_vote       5000 non-null   int64         
 9   verified_purchase  5000 non-null   bool          
 10  price              1159 non-null   float64       
 11  average_rating     5000 non-null   float64       
 12  rating_number      5000 non-null   int64         
 13  store              4714 non-null   object 

In [38]:
merged_df.isna().sum()

rating                  0
title                   0
text                    0
images                  0
asin                    0
parent_asin             0
user_id                 0
timestamp               0
helpful_vote            0
verified_purchase       0
price                3841
average_rating          0
rating_number           0
store                 286
dtype: int64

In [39]:
merged_df['user_id'].value_counts().value_counts().head()

count
1    1948
2     321
3     100
4      33
5      12
Name: count, dtype: int64

In [40]:
merged_df['clean_text'] = merged_df['text'].fillna('').astype(str)

merged_df['char_count'] = merged_df['clean_text'].str.len().astype(float)
merged_df['word_count'] = merged_df['clean_text'].str.split().str.len().astype(float)
merged_df['avg_word_length'] = merged_df['char_count'] / (merged_df['word_count'] + 1e-5)
merged_df['uppercase_ratio'] = merged_df['clean_text'].fillna('').apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1e-5)
)
merged_df['exclamation_count'] = merged_df['clean_text'].fillna('').apply(lambda x: x.count('!'))

In [ ]:
# 2. Text & stylometric features (including title)
merged_df["clean_text"] = (
    merged_df["title"].fillna("").astype(str)
    + " "
    + merged_df["text"].fillna("").astype(str)
).str.strip()

merged_df["char_count"] = merged_df["clean_text"].str.len().astype(float)
merged_df["word_count"] = merged_df["clean_text"].str.split().str.len().astype(float)
merged_df["avg_word_length"] = merged_df["char_count"] / (
    merged_df["word_count"] + 1e-5
)
merged_df["uppercase_ratio"] = merged_df["clean_text"].apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1e-5)
)
merged_df["exclamation_count"] = merged_df["clean_text"].apply(
    lambda x: x.count("!")
)

# 3. External/behavioral & item-metadata features
merged_df["is_extreme_rating"] = merged_df["rating"].isin([1, 5]).astype(int)
merged_df["verified_purchase"] = merged_df["verified_purchase"].astype(int)
user_counts = merged_df["user_id"].map(merged_df["user_id"].value_counts())
merged_df["is_cold_start_user"] = (user_counts == 1).astype(int)

merged_df["price_is_missing"] = merged_df["price"].isna().astype(int)
merged_df["store_is_missing"] = merged_df["store"].isna().astype(int)
merged_df["price_cleaned"] = merged_df["price"].fillna(0).astype(float)

stylometric_cols = [
    "char_count",
    "word_count",
    "avg_word_length",
    "uppercase_ratio",
    "exclamation_count",
]

external_cols = [
    "helpful_vote",
    "is_extreme_rating",
    "verified_purchase",
    "is_cold_start_user",
    "price_is_missing",
    "store_is_missing",
    "price_cleaned",
]

feature_cols = stylometric_cols + external_cols

In [43]:
print(merged_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   rating              5000 non-null   int64         
 1   title               5000 non-null   object        
 2   text                5000 non-null   object        
 3   images              5000 non-null   object        
 4   asin                5000 non-null   object        
 5   parent_asin         5000 non-null   object        
 6   user_id             5000 non-null   object        
 7   timestamp           5000 non-null   datetime64[ns]
 8   helpful_vote        5000 non-null   int64         
 9   verified_purchase   5000 non-null   int64         
 10  price               1159 non-null   float64       
 11  average_rating      5000 non-null   float64       
 12  rating_number       5000 non-null   int64         
 13  store               4714 non-null   object      

In [44]:
import numpy as np

# Condition 1: Extreme rating + Unverified + Zero helpful votes
cond1 = (merged_df['is_extreme_rating'] == 1) & (merged_df['verified_purchase'] == 0) & (merged_df['helpful_vote'] == 0)

# Condition 2: Extreme rating from a brand new/cold-start user with very short text
cond2 = (merged_df['is_extreme_rating'] == 1) & (merged_df['is_cold_start_user'] == 1) & (merged_df['char_count'] < 40)

# Condition 3: Suspiciously repetitive or low-effort formatting (extreme rating + low word count)
cond3 = (merged_df['is_extreme_rating'] == 1) & (merged_df['word_count'] < 5)

# Multi-signal scoring: flag as 1 if multiple suspicious markers align
score = cond1.astype(int) + cond2.astype(int) + cond3.astype(int)
y = (score >= 2).astype(int)

print("Target Class Distribution:")
print(y.value_counts())
print(f"\nPositive Class Ratio: {y.mean() * 100:.2f}%")

Target Class Distribution:
0    4806
1     194
Name: count, dtype: int64

Positive Class Ratio: 3.88%


In [45]:
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Stratified 80/20 split on row indices
indices = np.arange(len(merged_df))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# 2. Compute cold-start counts strictly from training users (defaulting unseen/cold-start to 1)
train_user_counts = merged_df.iloc[train_idx]['user_id'].value_counts()
cold_start_all = (
    merged_df['user_id'].map(train_user_counts).fillna(1).astype(int) == 1
).astype(int)

dense_df = merged_df[feature_cols].copy()
dense_df['is_cold_start_user'] = cold_start_all

# 3. Fit TF-IDF on training text only, then transform test text
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_text_train = tfidf.fit_transform(merged_df.iloc[train_idx]['clean_text'])
X_text_test = tfidf.transform(merged_df.iloc[test_idx]['clean_text'])

# 4. Fit StandardScaler on training dense features only
scaler = StandardScaler()
X_dense_train = scaler.fit_transform(dense_df.iloc[train_idx])
X_dense_test = scaler.transform(dense_df.iloc[test_idx])

# 5. Assemble final train/test feature matrices and targets
X_train = hstack([X_text_train, csr_matrix(X_dense_train)]).tocsr()
X_test_base = hstack([X_text_test, csr_matrix(X_dense_test)]).tocsr()

y_train = y.iloc[train_idx] if hasattr(y, 'iloc') else y[train_idx]
y_test = y.iloc[test_idx] if hasattr(y, 'iloc') else y[test_idx]

print(
    'Leak-free shapes -> X_train:',
    X_train.shape,
    '| X_test_base:',
    X_test_base.shape,
)

Leak-free shapes -> X_train: (4000, 5011) | X_test_base: (1000, 5011)


In [46]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

model = LogisticRegression(
    class_weight='balanced', max_iter=1000, random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test_base)
y_prob = model.predict_proba(X_test_base)[:, 1]

print(f'Baseline F1 Score: {f1_score(y_test, y_pred):.4f}')
print(f'Baseline ROC-AUC:  {roc_auc_score(y_test, y_prob):.4f}')

Baseline F1 Score: 0.4634
Baseline ROC-AUC:  0.9795


In [47]:
rng = np.random.default_rng(42)
dropout_rates = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
n_text = X_text_train.shape[1]
results = []

for rate in dropout_rates:
  X_masked = X_test_base.tolil()
  dense_block = X_masked[:, n_text:].toarray()

  if rate > 0:
    mask = rng.random(dense_block.shape) < rate
    dense_block[mask] = 0.0

  X_masked[:, n_text:] = dense_block
  X_eval = X_masked.tocsr()

  preds = model.predict(X_eval)
  probs = model.predict_proba(X_eval)[:, 1]

  results.append({
      'dropout_rate': f'{int(rate * 100)}%',
      'f1': round(f1_score(y_test, preds), 4),
      'roc_auc': round(roc_auc_score(y_test, probs), 4),
  })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

dropout_rate     f1  roc_auc
          0% 0.4634   0.9795
         20% 0.3600   0.9009
         40% 0.1875   0.8523
         60% 0.0870   0.8072
         80% 0.0500   0.7699
        100% 0.0000   0.7775
